In [1]:
import duckdb, time
import pandas as pd
from pathlib import Path

SEED = 2026
con = duckdb.connect()
PROCESSED = Path("..") / "data" / "processed"

def load_sample(name, n):
    path = (PROCESSED / f"{name}.csv").as_posix()
    return con.execute(f"""
        SELECT id, body_norm, recommendation_status, rating_band
        FROM read_csv('{path}', header=true, all_varchar=true)
        USING SAMPLE reservoir({n} ROWS) REPEATABLE ({SEED})
    """).df()

train_s = load_sample("train_supervised", 300000)
val_s = load_sample("validation", 100000)
print(len(train_s), len(val_s))
print(train_s["recommendation_status"].value_counts(dropna=False))

300000 100000
recommendation_status
recommended        197087
NaN                 51085
no_idea             28030
not_recommended     23798
Name: count, dtype: int64


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, balanced_accuracy_score

tr = train_s.dropna(subset=["recommendation_status", "body_norm"])
va = val_s.dropna(subset=["recommendation_status", "body_norm"])

t0 = time.time()
vec = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=3,
                      max_features=300000, sublinear_tf=True,
                      token_pattern=r"(?u)\b\w+\b")
X_tr = vec.fit_transform(tr["body_norm"])
X_va = vec.transform(va["body_norm"])
print("matrix:", X_tr.shape, "seconds:", round(time.time() - t0, 1))

t0 = time.time()
clf = LogisticRegression(C=1.0, class_weight="balanced", max_iter=300)
clf.fit(X_tr, tr["recommendation_status"])
print("fit seconds:", round(time.time() - t0, 1))

pred = clf.predict(X_va)
print("macro F1:", round(f1_score(va["recommendation_status"], pred, average="macro"), 4))
print("balanced acc:", round(balanced_accuracy_score(va["recommendation_status"], pred), 4))
print(classification_report(va["recommendation_status"], pred, digits=3))

matrix: (248915, 111933) seconds: 8.6
fit seconds: 27.5
macro F1: 0.6551
balanced acc: 0.7071
                 precision    recall  f1-score   support

        no_idea      0.334     0.533     0.411      9593
not_recommended      0.590     0.744     0.658      8269
    recommended      0.955     0.844     0.896     67465

       accuracy                          0.799     85327
      macro avg      0.627     0.707     0.655     85327
   weighted avg      0.850     0.799     0.818     85327



In [5]:
results = {}

def run_model(name, vec):
    t0 = time.time()
    X_tr = vec.fit_transform(tr["body_norm"])
    X_va = vec.transform(va["body_norm"])
    print(name, "matrix:", X_tr.shape, "seconds:", round(time.time() - t0, 1))
    t0 = time.time()
    clf = LogisticRegression(C=1.0, class_weight="balanced", max_iter=300)
    clf.fit(X_tr, tr["recommendation_status"])
    print("fit seconds:", round(time.time() - t0, 1))
    pred = clf.predict(X_va)
    f1s = f1_score(va["recommendation_status"], pred, average=None,
                   labels=["no_idea", "not_recommended", "recommended"])
    results[name] = {
        "macro_f1": round(f1_score(va["recommendation_status"], pred, average="macro"), 4),
        "balanced_acc": round(balanced_accuracy_score(va["recommendation_status"], pred), 4),
        "f1_no_idea": round(f1s[0], 3),
        "f1_not_recommended": round(f1s[1], 3),
        "f1_recommended": round(f1s[2], 3),
        "n_features": X_tr.shape[1],
    }

run_model("word_1_2", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=3,
                                      max_features=300000, sublinear_tf=True,
                                      token_pattern=r"(?u)\b\w+\b"))
run_model("char_2_5", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=5,
                                      max_features=300000, sublinear_tf=True))
pd.DataFrame(results).T

word_1_2 matrix: (248915, 111933) seconds: 9.3
fit seconds: 27.2
char_2_5 matrix: (248915, 75109) seconds: 28.1
fit seconds: 32.7


,macro_f1,balanced_acc,f1_no_idea,f1_not_recommended,f1_recommended,n_features
word_1_2,0.6551,0.7071,0.411,0.658,0.896,111933.0
char_2_5,0.6338,0.6994,0.393,0.632,0.877,75109.0


In [6]:
from sklearn.metrics import confusion_matrix

band_order = ["low", "medium", "high"]
band_code = {"low": 0, "medium": 1, "high": 2}

trb = train_s.dropna(subset=["rating_band", "body_norm"])
vab = val_s.dropna(subset=["rating_band", "body_norm"])
print(len(trb), len(vab))
print(trb["rating_band"].value_counts())

t0 = time.time()
vec_b = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=3,
                        max_features=300000, sublinear_tf=True,
                        token_pattern=r"(?u)\b\w+\b")
Xb_tr = vec_b.fit_transform(trb["body_norm"])
Xb_va = vec_b.transform(vab["body_norm"])
print("matrix:", Xb_tr.shape, "seconds:", round(time.time() - t0, 1))

t0 = time.time()
clf_b = LogisticRegression(C=1.0, class_weight="balanced", max_iter=300)
clf_b.fit(Xb_tr, trb["rating_band"])
print("fit seconds:", round(time.time() - t0, 1))

pred_b = clf_b.predict(Xb_va)
print("macro F1:", round(f1_score(vab["rating_band"], pred_b, average="macro"), 4))
print("balanced acc:", round(balanced_accuracy_score(vab["rating_band"], pred_b), 4))
print(classification_report(vab["rating_band"], pred_b, labels=band_order, digits=3))

cm = confusion_matrix(vab["rating_band"], pred_b, labels=band_order)
print(pd.DataFrame(cm, index=[f"true_{b}" for b in band_order],
                   columns=[f"pred_{b}" for b in band_order]))

dist = (vab["rating_band"].map(band_code)
        - pd.Series(pred_b, index=vab.index).map(band_code)).abs()
print("ordinal distance error:", round(dist.mean(), 4))
print("share of distance-2 errors:", round((dist == 2).mean(), 4))

275975 91279
rating_band
high      197133
medium     51829
low        27013
Name: count, dtype: int64
matrix: (275975, 119692) seconds: 9.8
fit seconds: 32.0
macro F1: 0.6213
balanced acc: 0.6582
              precision    recall  f1-score   support

         low      0.551     0.735     0.630      9018
      medium      0.366     0.462     0.408     17323
        high      0.880     0.778     0.826     64938

    accuracy                          0.713     91279
   macro avg      0.599     0.658     0.621     91279
weighted avg      0.750     0.713     0.727     91279

             pred_low  pred_medium  pred_high
true_low         6628         1688        702
true_medium      3128         8004       6191
true_high        2272        12173      50493
ordinal distance error: 0.3191
share of distance-2 errors: 0.0326


In [7]:
nbytes = Xb_tr.data.nbytes + Xb_tr.indices.nbytes + Xb_tr.indptr.nbytes
print("rows:", Xb_tr.shape[0])
print("non-zero entries:", Xb_tr.nnz)
print("average per review:", round(Xb_tr.nnz / Xb_tr.shape[0], 1))
print("matrix size in MB:", round(nbytes / 1024 ** 2, 1))

rows: 275975
non-zero entries: 4370477
average per review: 15.8
matrix size in MB: 51.1


In [9]:
import psutil, gc

def mem_gb():
    return round(psutil.Process().memory_info().rss / 1024 ** 3, 2)

print("memory before:", mem_gb(), "GB")
train_big = load_sample("train_supervised", 1000000)
tr1 = train_big.dropna(subset=["recommendation_status", "body_norm"])
print("rows:", len(tr1), "| memory:", mem_gb(), "GB")

t0 = time.time()
vec1 = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=3,
                       max_features=300000, sublinear_tf=True,
                       token_pattern=r"(?u)\b\w+\b")
X1_tr = vec1.fit_transform(tr1["body_norm"])
X1_va = vec1.transform(va["body_norm"])
print("matrix:", X1_tr.shape, "seconds:", round(time.time() - t0, 1),
      "| memory:", mem_gb(), "GB")

t0 = time.time()
clf1 = LogisticRegression(C=1.0, class_weight="balanced", max_iter=300)
clf1.fit(X1_tr, tr1["recommendation_status"])
print("fit seconds:", round(time.time() - t0, 1), "| memory:", mem_gb(), "GB")

pred1 = clf1.predict(X1_va)
print("macro F1:", round(f1_score(va["recommendation_status"], pred1, average="macro"), 4))
print("balanced acc:", round(balanced_accuracy_score(va["recommendation_status"], pred1), 4))
print(classification_report(va["recommendation_status"], pred1, digits=3))

memory before: 1.34 GB
rows: 829591 | memory: 1.45 GB
matrix: (829591, 289869) seconds: 24.4 | memory: 1.45 GB


D:\maktab\145\digikala-review-classifier\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


fit seconds: 113.4 | memory: 1.45 GB
macro F1: 0.6652
balanced acc: 0.7181
                 precision    recall  f1-score   support

        no_idea      0.337     0.558     0.420      9593
not_recommended      0.616     0.753     0.678      8269
    recommended      0.959     0.844     0.898     67465

       accuracy                          0.803     85327
      macro avg      0.637     0.718     0.665     85327
   weighted avg      0.856     0.803     0.823     85327

